In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any, ClassVar

In [2]:
@dataclass
class Performer:
    """A performer in the Yes-And game."""
    
    # Instance fields (dataclass will auto-generate __init__)
    name: str
    role: str  # "host" or "player"
    client_config: Dict[str, str]  # Store config instead of client
    model: str
    system_prompt: str = ""
    temperature: float = 0.7
    max_tokens: int = 500

    client: OpenAI = field(init=False, default=None)
    
    def __post_init__(self):
        """Create client from stored config."""
        if self.client_config.get('base_url'):
            self.client = OpenAI(api_key=self.client_config['api_key'], base_url=self.client_config['base_url'])
        else:
            self.client = OpenAI(api_key=self.client_config.get('api_key'))
    
    # Shared across ALL Performer instances
    full_history: ClassVar[List[Dict[str, str]]] = []

    def set_system_prompt(self, system_prompt: str) -> None:
        """Reset or update the system prompt."""
        self.system_prompt = system_prompt

    def set_starting_message(self, starting_message: str) -> None:
        """Return the starting message for the performer."""
        msg = f"{self.name} says: {starting_message}"
        Performer.full_history = [{"role": "assistant", "content": msg}]

    def _chat(self, extra_user_msg: Optional[str] = None) -> str:
        """Internal helper to call the model with full history and append response."""
        history: List[Dict[str, str]] = [{"role": "system", "content": self.system_prompt}]
        history.extend(Performer.full_history)
        if extra_user_msg:
            history.append({"role": "user", "content": extra_user_msg})

        resp = self.client.chat.completions.create(
            model=self.model,
            messages=history,
            temperature=self.temperature,
            max_tokens=self.max_tokens
        )
        content = resp.choices[0].message.content

        # Handle None content
        if content is None:
            content = "[No response generated]"

        # Label with performer identity (so others know who said it)
        # Only add label if it's not already present
        if content.startswith(f"{self.name} says:"):
            labeled_content = content
        else:
            labeled_content = f"{self.name} says: {content}"

        # Save assistant reply into shared history
        Performer.full_history.append({"role": "assistant", "content": labeled_content})
        return labeled_content
    
    def user_interaction(self, user_message: str) -> str:
        """Continue conversation with full history (host + user)."""
        Performer.full_history.append({"role": "user", "content": user_message})
        return self._chat()

    def decide(self) -> str:
        """
        Host decides if the game should continue or end.
        By default, let the model make the decision based on the history.
        """
        assert self.role == "host", "Only the host should decide."
        decision_prompt = (
            "Based on the scene so far, decide whether to CONTINUE or END GAME. "
            "Reply with only 'continue' or 'end'."
        )
        decision = self._chat(extra_user_msg=decision_prompt).lower()
        
        # Remove the decision message from history (it's not part of the conversation)
        # _chat() just added it, so we can safely pop it
        if Performer.full_history:
            Performer.full_history.pop()

        if "end" in decision:
            return "end"
        return "continue" 
    
    def speak(self) -> str:
        """
        Player speaks by building on the chat history.
        Example: Wayne uses Ryan's full scene history as context for his next move.
        """
        # Be very explicit about only speaking as this one performer
        extra_prompt = (
            f"It is now YOUR turn to speak. You are {self.name}. "
            f"Respond ONLY as {self.name}. Do NOT speak for any other character. "
            f"Continue the scene with 2-3 sentences that accept what has been established and add something new."
        )
        return self._chat(extra_user_msg=extra_prompt)

    @classmethod
    def get_full_history(cls) -> List[Dict[str, str]]:
        """Return the full conversation so far."""
        return cls.full_history
    
    @classmethod
    def clear_full_history(cls) -> None:
        """Reset the shared history."""
        cls.full_history = []

In [20]:
load_dotenv(override=True)

# create client configurations for different models
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

# instantiate performers with client configs instead of client objects
host = Performer(
    name="Drew",
    role="host", 
    client_config={"api_key": os.getenv('OPENAI_API_KEY')}, 
    model="gpt-4o-mini")
    
performer1 = Performer(
    name="Ryan", 
    role="player", 
    client_config={"api_key": os.getenv('GOOGLE_API_KEY'), "base_url": gemini_url}, 
    model="gemini-2.5-flash")

performer2 = Performer(
    name="Wayne", 
    role="player", 
    client_config={"api_key": os.getenv('ANTHROPIC_API_KEY'), 
    "base_url": anthropic_url}, 
    model="claude-3-5-haiku-latest")

In [4]:
host_system_prompt = f"""You are {host.name}, the host of a multi-agent “Yes, And” improv game.You do not play the game yourself—you only guide it, 
moderate it, and make decisions.
Responsibilities
Solicit Scenario
Ask the user (audience) for a fun scenario to start the game.If unclear/inappropriate, ask them to rephrase once.
Frame the Scene
Convert the scenario into a structured Scene Brief (setting, tone, and constraints).
Broadcast this Scene Brief as instructions to the players. The instructions for {performer1.name} and {performer2.name}
should not be more than 3 sentences. You should not make up names for performers. It is there story to tell. Just 
give them the scene brief and let them play.
To start the game, say [HOST DECISION: Start Game] to start the game loop.
The Game Loop
Alternate turns between {performer1.name} (Player 1) and {performer2.name} (Player 2).
After each pair of turns, decide whether to continue or end.
If ending, wrap up with a closing message to the user.
Decision Making
Say [HOST DECISION: Start Game] to start the game. Only say this once to start the game loop
Say [HOST DECISION: Continue Game] to keep the game going.
Say [HOST DECISION: End Game] to stop the game."""

# Original performer promts. They had issues so replaced below. Keeping here for reference.
performer1_system_prompt = f"""You are {performer1.name}, Player 1 in the improv game Yes, And.
You play inside the scene brief that {host.name} (the host) provides.
Responsibilities
Always accept what has been established (the “Yes”).Always add something new that pushes the story forward (the “And”).
Stay within the tone, rules, and constraints that {host.name} defines.
Write 2–3 sentences per turn (unless {host.name} specifies otherwise).
Never act as {host.name} or {performer2.name} — only roleplay your own turn.
Do not start your turn with anything like "Performer says:" where performer can be any name. 
The audience knows who you are."""

performer2_system_prompt = f"""You are {performer2.name}, Player 2 in the improv game Yes, And.
You play inside the scene brief that {host.name} (the host) provides.
Responsibilities
Always accept what has been established (the “Yes”).Always add something new that pushes the story forward (the “And”).
Stay within the tone, rules, and constraints that {host.name} defines.
Write 2–3 sentences per turn (unless {host.name} specifies otherwise).
Never act as {host.name} or {performer1.name} — only roleplay your own turn.
Do not start your turn with anything like "Performer says:" where performer can be any name. 
The audience knows who you are."""

In [5]:
host.set_system_prompt(host_system_prompt)
# performer1.set_system_prompt(performer1_system_prompt)
# performer2.set_system_prompt(performer2_system_prompt)

In [6]:
# Update system prompts with stronger instructions to prevent multi-character responses
performer1_system_prompt_updated = f"""You are {performer1.name}, Player 1 in the improv game Yes, And.

CRITICAL RULES - READ CAREFULLY:
1. You are ONLY {performer1.name}. You do NOT play {performer2.name} or {host.name}.
2. When it's your turn, write ONLY your own dialogue and actions (2-3 sentences).
3. NEVER write dialogue or actions for {performer2.name}. They will speak separately.
4. STOP after your 2-3 sentences. Do not continue the scene beyond your turn.
5. Do not write "{performer1.name} says:" or "{performer2.name} says:" - just speak naturally.
6. You may give a scene appropriate name to the other performer if you do so in a natural way.
7. The other performer make give you a scene appropriate name. If they do, accept it as part of the scene.

Scene Rules:
- Play inside the scene brief that {host.name} provides
- Always accept what has been established (the "Yes")
- Always add something new that pushes the story forward (the "And")
- Stay within the tone, rules, and constraints {host.name} defines"""

performer2_system_prompt_updated = f"""You are {performer2.name}, Player 2 in the improv game Yes, And.

CRITICAL RULES - READ CAREFULLY:
1. You are ONLY {performer2.name}. You do NOT play {performer1.name} or {host.name}.
2. When it's your turn, write ONLY your own dialogue and actions (2-3 sentences).
3. NEVER write dialogue or actions for {performer1.name}. They will speak separately.
4. STOP after your 2-3 sentences. Do not continue the scene beyond your turn.
5. Do not write "{performer2.name} says:" or "{performer1.name} says:" - just speak naturally.
6. You may give a scene appropriate name to the other performer if you do so in a natural way.
7. The other performer make give you a scene appropriate name. If they do, accept it as part of the scene.

Scene Rules:
- Play inside the scene brief that {host.name} provides
- Always accept what has been established (the "Yes")
- Always add something new that pushes the story forward (the "And")
- Stay within the tone, rules, and constraints {host.name} defines"""

# Apply the updated prompts
performer1.set_system_prompt(performer1_system_prompt_updated)
performer2.set_system_prompt(performer2_system_prompt_updated)


In [7]:
print(host.name)
print(performer1.name)
print(performer2.name)

Drew
Ryan
Wayne


In [8]:
def format_transcript(history: List[Dict[str, str]], host: Performer, p1: Performer, p2: Performer) -> str:
    """
    Turn shared history into a single HTML transcript with color coding.
    
    Args:
        history: The conversation history
        host: The host Performer
        p1: First player Performer
        p2: Second player Performer
    
    Returns:
        HTML formatted transcript string
    """
    # Map performers to their styling (color, emoji)
    performer_styles = {
        host.name: {
            "color": "#2563eb",  # Blue for host
            "emoji": "🎭"
        },
        p1.name: {
            "color": "#059669",  # Green for player 1
            "emoji": "🎪"
        },
        p2.name: {
            "color": "#7c3aed",  # Purple for player 2
            "emoji": "🎨"
        }
    }
    
    lines = []
    for msg in history:
        if msg["role"] == "user":
            lines.append(f"<p><strong>Audience:</strong> {msg['content']}</p>")
        else:
            # assistant lines are already labeled like "Name says: ..."
            content = msg["content"]
            
            # Add color coding based on performer name
            for performer_name, style in performer_styles.items():
                name_prefix = f"{performer_name} says:"
                if name_prefix in content:
                    styled_prefix = f'<span style="color: {style["color"]}; font-weight: bold;">{style["emoji"]} {name_prefix}</span>'
                    content = content.replace(name_prefix, styled_prefix)
                    break
           
            lines.append(f"<p>{content}</p>")
    
    # Wrap in a div with some styling
    return f'<div style="font-family: Arial, sans-serif; line-height: 1.6;">{"".join(lines)}</div>'

In [9]:
def run_yes_and_game_stream(host: Performer, p1: Performer, p2: Performer, 
                             max_rounds: int = 100):
    """
    Generator that yields formatted HTML transcript during game phase only.
    Assumes chat phase has already populated full_history with host/user interaction.
    """
    # Game starts - performers alternate speaking
    players = [p1, p2]
    turn = 0
    
    for _ in range(max_rounds):
        # Current player speaks
        current = players[turn % 2]
        current.speak()
        yield format_transcript(Performer.get_full_history(), host, p1, p2)
        
        # Host decides whether to continue or end
        decision = host.decide()
        yield format_transcript(Performer.get_full_history(), host, p1, p2)
        
        if decision == "end":
            # Host gives closing message
            host.user_interaction(
                "Please give a short 1-2 sentence closing line of wisdom for the players and audience."
            )
            yield format_transcript(Performer.get_full_history(), host, p1, p2)
            break
        
        turn += 1

In [10]:
def chat_with_host(user_message: str, chat_history: List[Dict[str, str]], 
                   host: Performer) -> tuple[List[Dict[str, str]], bool]:
    """Handle chat interaction between user and host."""
    if not user_message.strip():
        return chat_history, False
    
    host_response = host.user_interaction(user_message)
    chat_history.append({"role": "user", "content": user_message})
    chat_history.append({"role": "assistant", "content": host_response})
    # Check if host decided to start game
    game_should_start = "[HOST DECISION: Start Game]".upper() in host_response.upper()
    return chat_history, game_should_start


In [11]:
def initialize_game(host: Performer, p1: Performer, p2: Performer) -> List[Dict[str, str]]:
    """Initialize new game session."""
    Performer.clear_full_history()
    host.set_starting_message(
        f"Hello, I am {host.name} and I am the host of the Yes-And game. "
        f"Please tell me what scenario you'd like {p1.name} and {p2.name} to perform!"
    )
    # Return starting message for display
    return [{"role": "assistant", "content": Performer.get_full_history()[0]["content"]}]


In [12]:
with gr.Blocks(title="Yes, And — Interactive Game") as demo:
    gr.Markdown("# 🎭 Yes, And — Multi-Agent Improv")
    gr.Markdown("Chat with the host to set up your scenario, then watch the performers improvise!")
    
    # State for tracking game progress (Performer objects referenced from outer scope)
    game_started = gr.State(False)
    
    with gr.Row():
        with gr.Column(scale=1):
            # Chat phase controls
            chatbot = gr.Chatbot(label="Chat with Host", height=400, type="messages")
            msg_input = gr.Textbox(
                label="Your message",
                placeholder="Describe the scenario you'd like...",
                lines=2
            )
            with gr.Row():
                send_btn = gr.Button("Send", variant="primary")
                clear_btn = gr.Button("New Game")
            
            max_rounds_slider = gr.Slider(
                5, 10, value=6, step=1, 
                label="Max Rounds"
            )
        
        with gr.Column(scale=1):
            # Game phase display
            game_display = gr.HTML(
                label="Game Transcript",
                value="<p><em>Game transcript will appear here after host starts the game...</em></p>"
            )
    
    # Event handlers (reference performers from outer scope)
    def handle_message(user_msg, history, game_state):
        """Process user message and check if game should start."""
        if game_state:  # Game already started
            return history, game_state, gr.update()
        
        updated_history, should_start = chat_with_host(user_msg, history, host)
        
        if should_start:
            # Trigger game stream
            return updated_history, True, gr.update(interactive=False)
        
        return updated_history, game_state, gr.update()
    
    def start_game_stream(game_state, max_rounds):
        """Start streaming the game if game has started."""
        if not game_state:
            return "<p><em>Game transcript will appear here after host starts the game...</em></p>"
        
        for transcript in run_yes_and_game_stream(host, performer1, performer2, max_rounds):
            yield transcript
    
    def reset_game():
        """Reset for new game."""
        initial_chat = initialize_game(host, performer1, performer2)
        return (
            initial_chat,  # chatbot
            False,  # game_started
            gr.update(interactive=True),  # msg_input
            "<p><em>Game transcript will appear here after host starts the game...</em></p>"  # game_display
        )
    
    # Send message -> update chat -> potentially start game stream
    send_click = send_btn.click(
        fn=handle_message,
        inputs=[msg_input, chatbot, game_started],
        outputs=[chatbot, game_started, msg_input]
    ).then(
        fn=lambda: "", 
        inputs=None,
        outputs=[msg_input]
    ).then(
        fn=start_game_stream,
        inputs=[game_started, max_rounds_slider],
        outputs=[game_display]
    )
    
    # Clear button
    clear_btn.click(
        fn=reset_game,
        inputs=None,
        outputs=[chatbot, game_started, msg_input, game_display]
    )
    
    # Initialize on load
    demo.load(
        fn=reset_game,
        inputs=None,
        outputs=[chatbot, game_started, msg_input, game_display]
    )

demo.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [13]:
for item in Performer.get_full_history():
    print(item)

print(len(Performer.get_full_history()))
# print(Performer.get_full_history()[6])

0


In [14]:
# print(Performer.get_full_history()[6])

In [15]:
# print(host.user_interaction("How about a pair of dogs that can fly and fight crime."))


In [16]:
# print(performer1.speak())

In [17]:
# print(performer2.speak())
# print(host.decide())

In [18]:
# for item in Performer.get_full_history():
#     print(item)


In [19]:
# print(format_transcript(Performer.get_full_history(), host, performer1, performer2))